# デバッグ！！

In [1]:
import os
import sys
from pprint import pprint
from dotenv import load_dotenv
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from googleapiclient.discovery import build
from pytube import extract
from googleapiclient.discovery import Resource
from pathlib import Path
import json

In [2]:
# my_youtube
# コメントデータのレスポンスから、キーに基づいて情報を抽出する
def extract_data(response: dict, keys: list):
    # データ用リスト
    data = []
    
    # データの抽出
    for item in response.get("items", []):
        comment = {k: item["snippet"]["topLevelComment"]["snippet"][k] for k in keys}
        data.append(comment)
    return data

# youtube動画のIDから、全てののコメント情報を取得する
def get_all_comments(video_id: str, youtube: Resource):
    # 抽出するデータのキー
    keys = ["textOriginal", "likeCount", "publishedAt"]  # コメントの投稿日時も入れたいなぁ
    
    # データ用リスト
    data = []
    # ページネーション用のオブジェクト
    next_page_token = None

    # テスト段階でリクエスト数を節約するためのカウンター変数
    count = 0
    while True:
        # APIリクエスト処理...
        count += 1
        
        # snippetのみを指定してクォータ消費を最小限に(1リクエスト=2点)
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,  # 1回で取れる最大数
            pageToken=next_page_token,
            textFormat="plainText"
        )
        response = request.execute()
        
        # データの抽出(それぞれが1つのデータを示す辞書が入ったリスト)
        subset_data = extract_data(response, keys)
        data += subset_data

        # 次のページがあるか確認
        next_page_token = response.get("nextPageToken")

        # 次のページがなければ、または5回(500件)に達したらループ終了
        # 動作確認ができたら、countを外せば全取得モードになる
        if not next_page_token or count >= 1:
            break

    return data

# youtube動画のIDから、該当動画の情報を取得する
def get_video_info(video_id: str, youtube: Resource):
    request = youtube.videos().list(
        part="snippet",
        id=video_id
    ).execute()

    # タイトル、説明文、チャンネル名、投稿時間
    keys = ["publishedAt", "title", "channelTitle"]
    v_info = {k: request["items"][0]["snippet"][k] for k in keys}
    
    return v_info

# コメント情報を辞書で入れているリストから、JSONLファイルで保存する
def save_jsonl(data: list, file_name: str):
    # # dataディレクトリ取得
    # BASE_DIR = get_root_dir()
    # DATA_DIR = Path(BASE_DIR, "data")
    # # 安全のために、ディレクトリが存在しない場合に作成する処理を入れる
    # DATA_DIR.mkdir(exist_ok=True)
    
    # ファイルパス作成
    jsonl_path = Path(Path(file_name).with_suffix(".jsonl"))
    
    # JSONLファイルへの書き込み処理
    with open(jsonl_path, "w") as f:
        for entry in data:
            f.write(json.dumps(entry) + "\n")
            
    return jsonl_path

In [3]:
load_dotenv()

True

In [4]:
youtube_api_key = os.getenv("YOUTUBE_API_KEY")
google_cloud_project = os.getenv("GOOGLE_CLOUD_PROJECT")
google_cloud_location = os.getenv("GOOGLE_CLOUD_LOCATION")

In [5]:
vertexai.init(
    project=google_cloud_project,
    location=google_cloud_location
)

In [6]:
# my_youtube
# APIクライアントの構築
youtube = build("youtube", "v3", developerKey=youtube_api_key)

In [7]:
# video_id 取得
video_url = "https://www.youtube.com/watch?v=yXQViqx6GMY"
video_id = extract.video_id(video_url)
print(video_id)

yXQViqx6GMY


In [8]:
# コメントのサンプルを取得する
data = get_all_comments(video_id, youtube)

In [9]:
data

[{'textOriginal': '😊😊😊❤😊😊',
  'likeCount': 0,
  'publishedAt': '2026-03-11T15:49:41Z'},
 {'textOriginal': 'She needs to sneeze',
  'likeCount': 0,
  'publishedAt': '2026-03-11T11:26:19Z'},
 {'textOriginal': 'I fell in love with Mariah ever since I listened to her first song! Her voice comes from Heaven, and besides she is such a beautiful woman!',
  'likeCount': 0,
  'publishedAt': '2026-03-11T02:29:25Z'},
 {'textOriginal': 'All I want for Christmas is you. By Mariah Carey is my ultimate favourite Christmas song of all times',
  'likeCount': 0,
  'publishedAt': '2026-03-10T21:33:45Z'},
 {'textOriginal': 'I love this song',
  'likeCount': 3,
  'publishedAt': '2026-03-07T21:04:12Z'},
 {'textOriginal': 'How us see canada 😂',
  'likeCount': 0,
  'publishedAt': '2026-03-05T23:48:57Z'},
 {'textOriginal': 'Října se to 😢😊19.🥺😭♥️👭🫀😢😭🥃🥳🌝😊',
  'likeCount': 0,
  'publishedAt': '2026-03-04T10:31:07Z'},
 {'textOriginal': '-Cool- memories',
  'likeCount': 0,
  'publishedAt': '2026-02-28T10:47:27Z'},


In [10]:
len(data)

100

In [11]:
# コメント情報の入ったデータセットから、欲しい情報を抽出してJSONLファイルで保存する
jsonl_path = save_jsonl(data=data, file_name="comment_data")

In [12]:
# 動画自体の情報
v_info = get_video_info(video_id, youtube)

In [13]:
model = GenerativeModel("gemini-3-flash-preview")

D:\01_Business\projects\training\10_Summarize_Youtube_Comment\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [ ]:
# 2. Geminiによる要約
# summary_text = my_gemini.summarize_comments(jsonl_path, v_info)

In [29]:
prompt = my_gemini.make_prompt(v_info)
print(prompt)


    提供したJSONLファイルは、YouTubeのコメントデータです。
    該当するYoutube動画は、2009-11-24T06:21:35Zに
    「MariahCareyVEVO」チャンネルに投稿された「Mariah Carey - All I Want For Christmas Is You (Official Video)」という名前の動画です。
    JSONLファイルの各行は 「"textOriginal": "コメント内容",
    "likeCount": 高評価数, "publishedAt": コメントの投稿日時」 の形式になっています。
    これらを分析し、以下の点を1000文字以内でレポートしてください。
    1. 全体的な反応の傾向
    2. 特に高評価（likeCount）が多いコメントの共通点
    3. 目立つキーワードや話題
    


In [30]:
# ファイルの読み込みとPartオブジェクトの作成
with open(jsonl_path, "rb") as f:
    file_data = f.read()

In [34]:
file_data[:1000]

b'{"textOriginal": "\\ud83d\\ude0a\\ud83d\\ude0a\\ud83d\\ude0a\\u2764\\ud83d\\ude0a\\ud83d\\ude0a", "likeCount": 0, "publishedAt": "2026-03-11T15:49:41Z"}\r\n{"textOriginal": "She needs to sneeze", "likeCount": 0, "publishedAt": "2026-03-11T11:26:19Z"}\r\n{"textOriginal": "I fell in love with Mariah ever since I listened to her first song! Her voice comes from Heaven, and besides she is such a beautiful woman!", "likeCount": 0, "publishedAt": "2026-03-11T02:29:25Z"}\r\n{"textOriginal": "All I want for Christmas is you. By Mariah Carey is my ultimate favourite Christmas song of all times", "likeCount": 0, "publishedAt": "2026-03-10T21:33:45Z"}\r\n{"textOriginal": "I love this song", "likeCount": 3, "publishedAt": "2026-03-07T21:04:12Z"}\r\n{"textOriginal": "How us see canada \\ud83d\\ude02", "likeCount": 0, "publishedAt": "2026-03-05T23:48:57Z"}\r\n{"textOriginal": "\\u0158\\u00edjna se to \\ud83d\\ude22\\ud83d\\ude0a19.\\ud83e\\udd7a\\ud83d\\ude2d\\u2665\\ufe0f\\ud83d\\udc6d\\ud83e\\ud

In [35]:
# JSONLデータとしてPartを作成
file_part = Part.from_data(
    data=file_data,
    mime_type="application/json"
)

In [38]:
file_part

inline_data {
  mime_type: "application/json"
  data: "{\"textOriginal\": \"\\ud83d\\ude0a\\ud83d\\ude0a\\ud83d\\ude0a\\u2764\\ud83d\\ude0a\\ud83d\\ude0a\", \"likeCount\": 0, \"publishedAt\": \"2026-03-11T15:49:41Z\"}\r\n{\"textOriginal\": \"She needs to sneeze\", \"likeCount\": 0, \"publishedAt\": \"2026-03-11T11:26:19Z\"}\r\n{\"textOriginal\": \"I fell in love with Mariah ever since I listened to her first song! Her voice comes from Heaven, and besides she is such a beautiful woman!\", \"likeCount\": 0, \"publishedAt\": \"2026-03-11T02:29:25Z\"}\r\n{\"textOriginal\": \"All I want for Christmas is you. By Mariah Carey is my ultimate favourite Christmas song of all times\", \"likeCount\": 0, \"publishedAt\": \"2026-03-10T21:33:45Z\"}\r\n{\"textOriginal\": \"I love this song\", \"likeCount\": 3, \"publishedAt\": \"2026-03-07T21:04:12Z\"}\r\n{\"textOriginal\": \"How us see canada \\ud83d\\ude02\", \"likeCount\": 0, \"publishedAt\": \"2026-03-05T23:48:57Z\"}\r\n{\"textOriginal\": \"\\u015

In [39]:
response = model.generate_content([prompt, file_part])

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [ ]:
response.text